# SpatialMesh — Day 10: GNN Training
Train `SpatialMeshGAT` on the 8K snapshots with `spatialmesh_loss`.

**Key mechanics:**
- Repulsion warmup: high weight (0.5) epochs 0-9 to spread speakers, then normal (0.05)
- Track validation loss AND separability score (the MOS bridge)
- Hard-case ablation: report separability on hard vs easy scenes separately
- Save best checkpoint to Drive on every val improvement
- Fallback: if val loss plateaus high by epoch 20, stop and use rule-based

In [ ]:
# Cell 1 — Setup
!pip install -q torch_geometric
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, random, json, time
from pathlib import Path
from torch_geometric.nn import GATv2Conv
from google.colab import drive
drive.mount('/content/drive')

BASE='/content/drive/MyDrive/SpatialMesh/'
DATA_DIR=BASE+'gnn_data/'
SAVE_DIR=BASE+'models/'
import sys; sys.path.append(BASE+'src/')
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# Cell 2 — Import model + loss (from src/ on Drive)
from spatialmesh_gat import SpatialMeshGAT
import spatialmesh_loss as L
print('Model + loss imported.')
print('Loss weights:', L.W_INTERFERENCE, L.W_REPULSION,
      L.W_ELEVATION, L.W_COMFORT, L.W_STABILITY)

In [ ]:
# Cell 3 — Load all snapshots, move to device
files=sorted(Path(DATA_DIR).glob('snapshots_*.pt'))
print(f'{len(files)} files')
data=[]
for f in files:
    data.extend(torch.load(f))
print(f'Total snapshots: {len(data)}')

# Move tensors to device once (small dataset, fits in memory)
def to_dev(s):
    return {k:(v.to(DEVICE) if torch.is_tensor(v) else v)
            for k,v in s.items()}
data=[to_dev(s) for s in data]

# Train/val split
random.seed(42); random.shuffle(data)
n_val=1500
val_set=data[:n_val]; train_set=data[n_val:]
print(f'Train: {len(train_set)}  Val: {len(val_set)}')
n_hard=sum(s['meta']['hard_case'] for s in val_set)
print(f'Val hard cases: {n_hard}/{n_val}')

In [ ]:
# Cell 4 — Loss + separability helpers for a single snapshot
def snap_loss(model, s):
    pred=model(s['x'], s['edge_index'], s['edge_attr'])
    l,comp=L.spatialmesh_loss(
        pred, s['prev_positions'], s['embeddings'],
        s['activity_mask'], s['dominance'], s['overlap_duration'])
    return pred, l, comp

def snap_sep(pred, s):
    return L.perceptual_separability_score(
        pred, s['embeddings'], s['activity_mask'],
        s['dominance'], s['overlap_duration'])

@torch.no_grad()
def evaluate(model, dataset):
    model.eval()
    tot=0; sep_all=[]; sep_hard=[]; sep_easy=[]
    for s in dataset:
        pred,l,_=snap_loss(model,s)
        tot+=l.item()
        sp=snap_sep(pred,s)
        sep_all.append(sp)
        (sep_hard if s['meta']['hard_case'] else sep_easy).append(sp)
    return (tot/len(dataset), np.mean(sep_all),
            np.mean(sep_hard) if sep_hard else 0,
            np.mean(sep_easy) if sep_easy else 0)

In [ ]:
# Cell 5 — Training loop with repulsion warmup + checkpointing
model=SpatialMeshGAT().to(DEVICE)
opt=torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)

EPOCHS=60
BATCH=64
WARMUP_EPOCHS=10
best_val=float('inf')
history=[]

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
t0=time.time()

for ep in range(EPOCHS):
    # Repulsion warmup: high early to force spread, then normal
    L.W_REPULSION = 0.5 if ep < WARMUP_EPOCHS else 0.05

    model.train()
    random.shuffle(train_set)
    ep_loss=0; n_batches=0

    for bstart in range(0, len(train_set), BATCH):
        batch=train_set[bstart:bstart+BATCH]
        opt.zero_grad()
        batch_loss=0
        for s in batch:
            _,l,_=snap_loss(model,s)
            batch_loss=batch_loss+l
        batch_loss=batch_loss/len(batch)
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        ep_loss+=batch_loss.item(); n_batches+=1
    sched.step()

    # Validation
    vl, vsep, vhard, veasy = evaluate(model, val_set)
    tl=ep_loss/n_batches
    history.append({'ep':ep,'train':tl,'val':vl,'sep':vsep,
                    'sep_hard':vhard,'sep_easy':veasy})

    # Checkpoint on improvement (only after warmup, weights stable)
    flag=''
    if ep>=WARMUP_EPOCHS and vl<best_val:
        best_val=vl
        torch.save(model.state_dict(), SAVE_DIR+'gat_best.pt')
        flag=' *saved'

    if ep%2==0 or ep==EPOCHS-1:
        print(f'ep{ep:2d} train={tl:.4f} val={vl:.4f} '
              f'sep={vsep:.1f} (hard={vhard:.1f} easy={veasy:.1f})'
              f'{flag}')

print(f'\nDONE in {time.time()-t0:.0f}s. Best val loss: {best_val:.4f}')
torch.save(model.state_dict(), SAVE_DIR+'gat_final.pt')
with open(SAVE_DIR+'training_history.json','w') as f:
    json.dump(history,f,indent=2)

In [ ]:
# Cell 6 — Plot training curves
import matplotlib.pyplot as plt
h=history
eps=[x['ep'] for x in h]
fig,ax=plt.subplots(1,2,figsize=(13,4))
ax[0].plot(eps,[x['train'] for x in h],label='train')
ax[0].plot(eps,[x['val'] for x in h],label='val')
ax[0].axvline(WARMUP_EPOCHS,ls='--',c='gray',label='warmup end')
ax[0].set_title('Loss'); ax[0].legend(); ax[0].set_xlabel('epoch')
ax[1].plot(eps,[x['sep'] for x in h],label='all')
ax[1].plot(eps,[x['sep_hard'] for x in h],label='hard cases')
ax[1].plot(eps,[x['sep_easy'] for x in h],label='easy cases')
ax[1].set_title('Perceptual Separability (MOS bridge)')
ax[1].legend(); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.savefig(SAVE_DIR+'training_curves.png',dpi=120)
plt.show()
print('Saved training_curves.png')

In [ ]:
# Cell 7 — Inspect a few predictions vs baseline
model.load_state_dict(torch.load(SAVE_DIR+'gat_best.pt'))
model.eval()

def fixed_baseline(activity):
    """Equal-spacing baseline for comparison (Fix 10)."""
    base={2:[-0.25,0.25],3:[0.0,-0.67,0.67],4:[-0.25,0.25,-0.75,0.75]}
    n=int(activity.sum().item())
    azs=base.get(n,[0,0.5,-0.5,1.0])
    pos=torch.zeros(4,2)
    ai=[i for i in range(4) if activity[i]==1]
    for idx,sp in enumerate(ai):
        pos[sp,0]=azs[idx] if idx<len(azs) else 0
    return pos.to(DEVICE)

print('GNN vs fixed-baseline separability (5 hard val cases):')
hard=[s for s in val_set if s['meta']['hard_case']][:5]
for s in hard:
    with torch.no_grad():
        pred=model(s['x'],s['edge_index'],s['edge_attr'])
    gnn_sep=snap_sep(pred,s)
    base_sep=snap_sep(fixed_baseline(s['activity_mask']),s)
    print(f'  GNN={gnn_sep:.1f}  baseline={base_sep:.1f}  '
          f'gain={gnn_sep-base_sep:+.1f}')